# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR^2 clinical dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print metadata info
md = dataset.metadata
print(f"{md.name}: {md.description}")
print(f"Identifier: {md.identifier}")
print(f"Published: {md.datePublished}\n")


## 2. Data Overview
Review available record sets, fields, and their IDs.

### Inspecting Record Sets

The dataset contains structured tabulations of clinical, laboratory, and genetic features. Let's enumerate the available `recordSet` entities and their field IDs.

In [ ]:
# List all record sets and their details
record_sets = dataset.metadata.recordSet

# If no record sets directly in metadata, try dataset.record_sets property
if not record_sets:
    record_sets = dataset.record_sets

for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', rs['@id'])}")
    print(f"  Fields:")
    for f in rs.get('field', []):
        print(f"    - Field @id: {f['@id']}, name: {f.get('name', f['@id'])}")
    print()


## 3. Data Extraction
Load data from each record set into a pandas DataFrame.

#### Using `@id` references for extraction
You'll see `@id` shown in overview above for each record set and field. Use those IDs for querying.

In [ ]:
# Prepare dataframes for each record set
dfs = {}

# Collect all recordSet @ids
record_set_ids = []
for rs in record_sets:
    record_set_ids.append(rs['@id'])

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dfs[rs_id] = pd.DataFrame(records)

# Show available columns in each record set
for rs_id in record_set_ids:
    print(f"RecordSet @id: {rs_id} columns: {dfs[rs_id].columns.tolist()}")
    print(dfs[rs_id].head())


## 4. Exploratory Data Analysis (EDA)
Apply common processing steps: filtering, normalization, grouping and transformation.

For demonstration, select a numeric field (such as hormone level or age) using its `@id` for analysis.

In [ ]:
# Choose a record set and numeric field for EDA
# Example: Use Table 1 or Table 2 for clinical or hormone values

# Replace with actual @id found in record set listing above.
# For demo, suppose record_set_id and field_id:
record_set_id = record_set_ids[0]
df = dfs[record_set_id]

# Replace these with actual field @id as discovered above
numeric_field_id = df.columns[0] if len(df.columns) > 0 else None  # demo: first column
group_field_id = df.columns[1] if len(df.columns) > 1 else None  # demo: second column

# Example threshold
if numeric_field_id and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field (if string/categorical)
    if group_field_id and pd.api.types.is_string_dtype(df[group_field_id]):
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by '{group_field_id}':")
        print(grouped_df.head())
else:
    print("No numeric field found in record set for demonstration.")

## 5. Visualization
Visualize distributions or relationships between fields. Here we plot the selected numeric field, grouped by another field if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization of numeric field distribution
if numeric_field_id and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Grouped boxplot
    if group_field_id and pd.api.types.is_string_dtype(df[group_field_id]):
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field found for plotting.")

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR^2 clinical dataset using `mlcroissant`, referencing all data elements by their `@id`. You can extend analyses by selecting different record sets and fields by their `@id`, performing more advanced filtering or modeling, and combining clinical, laboratory, and genetic information for research or reproducibility.

*End of notebook. Modify record set and field @id values for deeper exploration as needed!*